# 02 — Data Analysis Report (Deliverable B)

All tasks B1.1–B4.3 on cleaned `master_train.csv`.

Run the script for a full markdown report:
```bash
python src/analysis.py
```
Output: `reports/B_analysis_report.md`

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.analysis import run_all, write_report, export_tables, load_master

results = run_all()
write_report(results, ROOT / 'reports' / 'B_analysis_report.md')
export_tables(results, ROOT / 'reports' / 'B_tables')
print('Zone types:', results['zone_types'])

## B1 — Demand patterns

### B1.1 Volume by zone

In [ ]:
display(results['B1.1'])
print('Top zone carries most demand; Ayat has fewer hours (late launch) — compare means carefully.')

### B1.2 Hour-of-day profile by zone type

In [ ]:
display(results['B1.2'])
df = load_master()
df['zone_type'] = df['zone'].map(results['zone_types'])
wd = df[(df['is_weekend']==0) & df['trips'].notna()]
prof = wd.groupby(['zone_type','hour'])['trips'].mean().unstack('hour')
ax = prof.T.plot(figsize=(10,5), title='Weekday hourly profile by zone type')
ax.set_xlabel('Hour (EAT)'); ax.set_ylabel('Mean trips')
plt.tight_layout(); plt.show()

### B1.3 Weekday vs weekend

In [ ]:
display(results['B1.3'])
standout = results['B1.3'].iloc[0]['zone']
d = df[df['zone']==standout].dropna(subset=['trips'])
dow = d.groupby('dow')['trips'].mean()
dow.index = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow.plot(kind='bar', title=f'Day-of-week shape — {standout}', figsize=(7,4))
plt.ylabel('Mean trips'); plt.tight_layout(); plt.show()

### B1.4 Trend

In [ ]:
b14 = results['B1.4']
print({k: b14[k] for k in b14 if k != 'weekly'})
b14['weekly'].plot(x='week_index', y='weekly_trips', figsize=(9,4), title='City-wide weekly trips')
plt.ylabel('Trips'); plt.tight_layout(); plt.show()

## B2 — Weather

### B2.1 Timezone check

In [ ]:
b21 = results['B2.1']
print('Temp peak hour (EAT):', b21['temp_peak_hour_eat'])
print('Corr shifts:', b21['rain_demand_corr_by_shift_h'])
print('Wrong UTC-as-local corr:', b21['corr_if_utc_treated_as_local'])
print(b21['interpretation'])

### B2.2 Rain effect by zone type

In [ ]:
display(results['B2.2'])
print('Ratio>1 => rainy hours busier than matched dry hours. Rain is not uniform across zone types.')

### B2.3 Rain dose-response

In [ ]:
display(results['B2.3'])
print('Values relative to rain_class=none. Look for saturation from moderate to heavy.')

## B3 — Events & calendar

In [ ]:
print('### B3.1 Holidays')
display(results['B3.1'])
display(results['B3.1_zone'])
print('### B3.2 Football windows')
display(results['B3.2'])
print('### B3.3 Event type ranking')
display(results['B3.3'])
print('### B3.4 Cancelled / unlisted')
display(results['B3.4']['cancelled_footprint'])
print(results['B3.4']['unlisted_spikes'])

## B4 — Operations & data quality

In [ ]:
print('B4.1', results['B4.1'])
display(results['B4.2']['by_zone'])
print('Late launch:', results['B4.2']['late_launch_zones'])
print(results['B4.2']['treatment'])
print('B4.3', results['B4.3'])